In [42]:
import pandas as pd
import numpy as np
import re

df_path = "current_address_cleaned_with_county_3.csv"
excel_path = "MTSP-Data-FY25.xlsx"

df = pd.read_csv(df_path)
mtsp = pd.read_excel(excel_path)

def norm_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()

def clean_name(s: str):
    if pd.isna(s):
        return np.nan
    s = str(s)
    s = re.sub(r"[\'\(\)\-\.\/]", " ", s)
    s = re.sub(r"\b(county|town|city)\b", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_city(s):
    if pd.isna(s):
        return np.nan
    return clean_name(s).lower()

if 'matched_city_confirmed' in df.columns:
    mask_grafton = norm_series(df['matched_city_confirmed']).eq('grafton')
    df.loc[mask_grafton, 'matched_state_confirmed'] = 'MA'

if 'Current Residence' in df.columns:
    mask_hydepark = norm_series(df['Current Residence']).eq('hyde park')
    df.loc[mask_hydepark, 'matched_city_confirmed'] = 'boston'

mask_dorchester = norm_series(df['matched_city_confirmed']).eq('dorchester')
df.loc[mask_dorchester, 'matched_city_confirmed'] = 'boston'

if 'County_Name_from_MTSP' not in df.columns:
    df['County_Name_from_MTSP'] = np.nan

city_to_county = {
    'cape coral':   'Lee County',
    'fort smith':   'Sequoyah County',
    'englewood':    'Sarasota County',
    'myrtle beach': 'Horry County',
    'menomonie':    'Dunn County',
}
mc_norm = norm_series(df['matched_city_confirmed'])
for city, county in city_to_county.items():
    m = mc_norm.eq(city)
    df.loc[m, 'County_Name_from_MTSP'] = county

mask_boston = norm_series(df['matched_city_confirmed']).eq('boston')
df.loc[mask_boston, 'matched_state_confirmed'] = 'MA'

mtsp['cleaned_county_town_name'] = mtsp['county_town_name'].apply(clean_name)
mtsp['stusps_up'] = mtsp['stusps'].astype(str).str.upper()
mtsp['ctn_norm'] = mtsp['cleaned_county_town_name'].astype(str).str.lower()

# (stusps, cleaned_county_town_name_norm) -> County_Name
mtsp_keyed = (
    mtsp.dropna(subset=['ctn_norm'])
        .drop_duplicates(subset=['stusps_up', 'ctn_norm'])
        .set_index(['stusps_up', 'ctn_norm'])['County_Name']
)

mask_has_state_city = df['matched_state_confirmed'].notna() & df['matched_city_confirmed'].notna()
mask_need_lookup = mask_has_state_city & df['County_Name_from_MTSP'].isna()

def lookup_county(row):
    st = str(row['matched_state_confirmed']).upper()
    city_key = norm_city(row['matched_city_confirmed'])
    if pd.isna(city_key):
        return np.nan
    try:
        return mtsp_keyed.loc[(st, city_key)]
    except KeyError:
        return np.nan

df.loc[mask_need_lookup, 'County_Name_from_MTSP'] = df.loc[mask_need_lookup].apply(lookup_county, axis=1)

unmatched_mask = mask_has_state_city & df['County_Name_from_MTSP'].isna()
unmatched_rows = df.loc[unmatched_mask]

if len(unmatched_rows) == 0:
    print("All records have been successfully matched with their state in MTSP.")
else:
    print(f"{len(unmatched_rows)} records could not be matched with their state in MTSP.")

if len(unmatched_rows) > 0:
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 240)
    cols_to_show = ['Current Residence', 'matched_state_confirmed', 'matched_county_confirmed', 'County_Name_from_MTSP']
    existing_cols = [c for c in cols_to_show if c in df.columns]
    print(unmatched_rows[existing_cols])

df.to_csv("current_address_cleaned_county_matched.csv", index=False)

All records have been successfully matched with their state in MTSP.


/var/folders/l9/r26f9dhx16qfdfwf8j9f08dc0000gn/T/ipykernel_1090/2222884613.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Lee County' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[m, 'County_Name_from_MTSP'] = county


In [43]:
county_counts = df['County_Name_from_MTSP'].value_counts(dropna=False)

print("Distribution of County ")
print(county_counts)

Distribution of County 
County_Name_from_MTSP
Middlesex County           461
Worcester County           314
Essex County               279
Norfolk County             229
Suffolk County             130
Plymouth County             90
Bristol County              85
Rockingham County           23
Hampshire County            22
Hillsborough County         21
Dukes County                17
Barnstable County           16
Cheshire County             10
Providence County            9
NaN                          7
Grafton County               3
Washington County            3
Franklin County              3
Capitol Planning Region      2
Berkshire County             2
Hampden County               2
Sequoyah County              2
York County                  1
Merrimack County             1
Horry County                 1
New York County              1
Sarasota County              1
Dunn County                  1
Newport County               1
Coos County                  1
Lee County              